# Part 3 · LTV through day 365

Every analysis behind the Part 3 decisions, in the order they were made. Logic lives in `src/sciplay/ltv/`; this notebook only calls it.

**The approach in one line:** the first 3 days of play → a validated forecast of revenue in the first 30 days → **× 2.49, an assumption** → LTV365 (gross USD, a labelled proxy).

**Contents**
1. Target validation: is there a 365-day label?
2. Which horizon can we learn from?
3. Split by install date
4. Early behaviour: why 3 days
5. From 30 days to 365: the multiplier
6. Features and leakage checks
7. Model selection on the training folds
8. Test results
9. Explanations (SHAP) and reason codes
10. Uncertainty and confidence
11. Players too old for the model: the update rule
12. The output table

In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display

from sciplay.config import REPO_ROOT, get_settings
from sciplay.data.loader import load_raw
from sciplay.ltv import analysis as A, checks, config as C, features as F, labels, split

pio.renderers.default = 'plotly_mimetype+notebook_connected'
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.width', 200)

pday = pd.read_parquet(get_settings().processed_dir / 'player_day.parquet')
pday['DATE'] = pd.to_datetime(pday['DATE'])
raw = load_raw(get_settings().raw_data_path)
registry = pd.read_csv(REPO_ROOT / 'docs' / 'column_registry.csv')
installs = split.install_dates(pday)
data_end = pday['DATE'].max()
sets = split.assign_sets(installs, data_end)
lab = labels.label_frame(pday, installs, data_end)
print(f'{installs.size} players, data {pday.DATE.min():%d %b} – {data_end:%d %b %Y}')

700 players, data 09 May – 04 Nov 2026


## 1. Target validation: is there a 365-day label?

The brief asks not to assume `D365_SPEND` is a label because of its name, and to test its availability, population, stability across snapshots, relationship to lifetime revenue and exposure to censoring.

In [2]:
A.target_validation(raw, installs, pday)

,test,what we found,verdict
0,Availability,"Filled on 407 of 4,245 rows, only on rows of players who have paid (21 players); never filled for a non-payer: 0 rows",Payers only
1,Population,0 of 700 players reach day 365; the oldest has 169 days of data,No mature outcome
2,Stability,Changes between snapshots for 12 of 21 payers,Not fixed
3,Relationship to revenue,Equals LIFETIME_REVENUE on 100% of filled rows,Duplicate column
4,Censoring,"Every player is censored, at 0–169 days of data",Fully censored


**Conclusion:** `D365_SPEND` is lifetime spend so far, not spend at day 365, and no player is old enough for a 365-day outcome. **There is no valid 365-day label, so it cannot be used to train.** Any 365-day figure from this sample is a labelled proxy.

## 2. Which horizon can we learn from?

A label is usable only when its whole window is inside the data (the data ends 4 Nov).

In [3]:
A.label_horizons(pday, installs, data_end)

,horizon_days,players_with_complete_label,payers,last_install_with_label
0,30,518,16,2026-10-06
1,90,322,15,2026-08-07
2,365,0,0,None


**Decision: learn from revenue in the first 30 days.** 518 players have a complete 30-day label (16 paid). **Not 90 or 365 days: too few players with a complete label, so train and test sets would be too small** (90 days: 322 players, a test cohort ending in early August with about 4–5 payers; 365 days: none).

## 3. Split by install date

Every player belongs to exactly one set, by install date. Train and test players have a complete 30-day label; inference players have at least 3 days of data but an incomplete month; younger players are cold start. The model that is tested is the model that ships (no refit on test players).

In [4]:
summary = lab.groupby(sets).agg(players=('complete', 'size'),
                                paid_by_day_3=('revenue_first_k', lambda s: int((s > 0).sum())),
                                paid_by_day_30=('revenue_first_30d', lambda s: int((s > 0).sum())))
summary['installs'] = installs.groupby(sets).agg(lambda s: f'{s.min():%d %b} – {s.max():%d %b}')
summary.loc[['train', 'test', 'inference', 'cold_start'], ['installs', 'players', 'paid_by_day_3', 'paid_by_day_30']]

,installs,players,paid_by_day_3,paid_by_day_30
set,,,,
train,09 May – 31 Jul,300,5,10
test,01 Aug – 06 Oct,218,4,6
inference,07 Oct – 02 Nov,132,1,0
cold_start,03 Nov – 04 Nov,50,0,0


Accepted risk: labels of late-July training players run into August, overlapping the test players' calendar time. Players never overlap.

## 4. Early behaviour: why 3 days

How much of a player's first-month revenue is visible after 3 or 7 days (train + test players)?

In [5]:
A.early_signal(pday, installs, sets.index[sets.isin(['train', 'test'])])

,first_k_days,players,paid_by_day_k,paid_by_day_30,day30_payers_already_paid,share_of_30d_revenue_known,avg_30d_revenue_if_paid_early,avg_30d_revenue_if_not
0,3,518,9,16,9 of 16,0.39,61.76,0.466
1,7,518,11,16,11 of 16,0.45,66.65,0.119


In [6]:
curve = A.revenue_curve(pday, installs, sets.index[sets == 'train'])
fig = go.Figure(go.Scatter(x=curve.index, y=curve.values, mode='lines+markers', line=dict(color='#6B3FA0', width=2),
                           hovertemplate='Day %{x}: %{y:.0%} of 30-day revenue<extra></extra>'))
fig.add_vline(x=C.K - 1, line_dash='dot', line_color='#6E6680', annotation_text='feature date (day 3)')
fig.update_layout(title='Share of first-30-day revenue earned by each day (training players)',
                  xaxis_title='Days since install', yaxis_title=None, yaxis_tickformat='.0%',
                  template='plotly_white', height=380, showlegend=False)
fig.show()

This curve is also used to compute **expected revenue by day d** for the ahead/behind-forecast tracking.

A quick out-of-time check with the simple rule (no tuning): fit on train, score test.

In [7]:
for k in (3, 7):
    t, info = A.rule_benchmark(pday, installs, sets, k)
    print(f"First {k} days: {info['train_early_payers']} training players had paid by day {k}; "
          f"test = {info['test_players']} players, {info['test_payers']} payers, ${info['test_actual_total']:.0f}")
    display(t)

First 3 days: 5 training players had paid by day 3; test = 218 players, 6 payers, $378


,approach,MAE_$,predicted_/_actual_total,MAE_payers_only_$
0,Baseline: 30-day revenue = revenue so far,1.15,0.34,41.7
1,Baseline: training average for everyone,3.04,0.80,61.7
2,"Rule: early spend x 1.11, else $0.73",1.80,0.79,39.6


First 7 days: 7 training players had paid by day 7; test = 218 players, 6 payers, $378


,approach,MAE_$,predicted_/_actual_total,MAE_payers_only_$
0,Baseline: 30-day revenue = revenue so far,1.09,0.37,39.7
1,Baseline: training average for everyone,3.04,0.80,61.7
2,"Rule: early spend x 1.72, else $0.13",1.01,0.71,32.0


**Decision: one model, from the first 3 days.** 7 days was considered first:
- 7 days carries more signal: 45% of 30-day revenue known vs 39% at day 3; 11 vs 9 of the 16 payers have paid; 7 training payers vs 5; and the 7-day rule beats both baselines on every measure.
- But 7 days leaves **all 100 November installs** (and 103 players in total) in cold start. The November cohort (100 installs in 4 days) looks deliberately placed to be scored with almost no history.
- 3 days scores **132 players instead of 79**, half of November among them. The 3-day rule is worse than "revenue so far" on error per player (MAE rewards predicting $0), but much better calibrated (predicted ÷ actual 0.79 vs 0.34) and better on payers; the model adds activity features the rule does not use.
- Train and test sizes do not depend on the window: they are set by the complete 30-day label.

**Why not predict days 3–7 instead of 30?** Only 2 of 518 players make a first purchase between day 3 and day 7, and reaching day 365 would need two assumed multipliers instead of one.

Players younger than 3 days (installed 3–4 Nov) are **cold start**: they get the new-player average, marked low confidence.

*How the shares are calculated:* pooled dollars, i.e. total revenue all players spent by day d ÷ the total they spent in days 0–29. Dollar-weighted, so big payers dominate. The table uses train + test players (39% by day 3); the curve uses training players only (43% by day 3).

## 5. From 30 days to 365: the multiplier

LTV365 = 30-day forecast × (1 + r + r² + … + r¹¹): each later 30-day block is worth r × the block before. Can r be estimated from the data? Month-to-month revenue for cohorts old enough to have both months:

In [8]:
total = pday.assign(R=pday['REVENUE'].fillna(0)).groupby('PLAYER_ID')['R'].sum().sort_values(ascending=False)
top2 = total.index[:2].tolist()
print('Top spenders:', {p: round(v) for p, v in total.head(2).items()}, f'= {total.head(2).sum() / total.sum():.0%} of all revenue')
for name, ex in [('All players', []), ('Without the top spender', top2[:1]), ('Without the top 2', top2)]:
    print(name)
    display(A.monthly_ratios(pday, installs, data_end, exclude=ex))

Top spenders: {33840: 2367, 39593: 1013} = 81% of all revenue
All players


,step,players,revenue_month_m,revenue_month_m+1,ratio
0,month 1 → 2,422,518.0,1473.0,2.84
1,month 2 → 3,322,1473.0,887.0,0.60
2,month 3 → 4,228,887.0,584.0,0.66
3,month 4 → 5,134,541.0,198.0,0.37


Without the top spender


,step,players,revenue_month_m,revenue_month_m+1,ratio
0,month 1 → 2,421,398.0,958.0,2.40
1,month 2 → 3,321,958.0,76.0,0.08
2,month 3 → 4,227,76.0,58.0,0.77
3,month 4 → 5,133,15.0,0.0,0.00


Without the top 2


,step,players,revenue_month_m,revenue_month_m+1,ratio
0,month 1 → 2,420,260.0,83.0,0.32
1,month 2 → 3,320,83.0,76.0,0.91
2,month 3 → 4,226,76.0,58.0,0.77
3,month 4 → 5,132,15.0,0.0,0.00


With all players the later steps look like 0.4–0.65, but **without the top two spenders, revenue after month 1 is under $100 a month in the whole sample** and the ratios jump around with no pattern. Those apparent ratios came from two players.

**Conclusion: r cannot be estimated from this sample. We assume r = 0.6, giving × 2.49, with sensitivity r = 0.5–0.7 (× 2.00–3.29). One multiplier for every player.** The month-1 → 2 rise also shows that month 1 can understate players who ramp up.

In [9]:
A.multiplier_table()

,r,multiplier,LTV365 for a $10 30-day forecast
0,0.5,2.00,20.00
1,0.6,2.49,24.95
2,0.7,3.29,32.87


## 6. Features and leakage checks

Features come only from rows dated install … install + 2 and only from columns the registry allows, typed by how they should be summarised over the week.

In [10]:
spec = F.build_spec(registry, pday)
types = spec.as_table()
types['type'] = types['feature_type'].str.replace(r':.*', '', regex=True)
display(types['type'].value_counts().rename('columns').to_frame())
display(types[types['type'] == 'excluded'].groupby('feature_type')['column']
        .agg(columns='size', examples=lambda s: ', '.join(s.head(4))))

,columns
type,
excluded,111
count,96
state,14
flag,11
friction,10
average,8
categorical,6


,columns,examples
feature_type,,
excluded: constant or near constant in the first days,2,"INSTALL_FLAG, REACTIVATED"
"excluded: device, app & performance: describes the phone and app, not the player's intent to spend; with 6 payers it learned noise (used in the Part 4 friction list instead)",41,"DEVICE_HARDWARE, DEVICE_OS_VERSION, DEVICE_TYPE, END_APP_VERSION"
"excluded: file's own rolling window: duplicates our sums, window end not documented",45,"ACTIVE_30D_PAYER_DAU, ACTIVE_90D_PAYER_DAU, MAX_WEEKLY_SPEND, AVG_WAGER_LAST_7_ENGAGED"
"excluded: lifetime / consecutive counter: duplicates our sums, may be rewritten",12,"LIFETIME_PAYERS, END_LIFETIME_SPINS, START_LIFETIME_SPINS, LIFETIME_REVENUE"
excluded: not numeric,4,"ISLAND_HOPPING_ENDING_STATE, SOURCE, SOURCE_TYPE, F2P_USAGE"
excluded: recency: rebuilt from our own rows,7,"DAYS_SINCE_INSTALL, DAYS_SINCE_LAST_LOGIN, DAYS_SINCE_LAST_PURCHASE, DAYS_SINCE_LAST_SPIN"


| Type | Summary over days 0–2 |
|---|---|
| count | sum, mean per active day, max |
| state | value on the last day (+ change for levels) |
| average | mean over active days |
| friction | sum + any |
| flag | share of active days |
| categorical | first row; one-hot, levels with fewer than 20 training players → Other |
| built from our rows | revenue in days 0–2, active days, played on day 1, last active day, day of first purchase (n/a + never-paid flag), revenue per session, store conversion, purchase failure rate |

Ordered segment bins (`LEVEL_BIN`, `SPINNER_SEGMENT`, …) are not used: they are bucketed from measures already included (registry: use the source measures).

In [11]:
X = F.build_features(pday, installs, spec)
train_ids = sets.index[sets == 'train']
enc = F.CategoryEncoder(spec.categorical).fit(X.loc[train_ids])
Xt = enc.transform(X.loc[train_ids].drop(columns='feature_date'))
kept, dropped = F.prefilter(Xt, priority=['revenue_first_k', 'active_days'])
print(f'{X.shape[1] - 1} built features -> {Xt.shape[1]} after one-hot -> {len(kept)} candidates after the pre-filter')
display(dropped['dropped_because'].str.replace(r' with .*', '', regex=True).value_counts().rename('features').to_frame())
pd.Series({c: ', '.join(v) + ', Other' for c, v in enc.levels_.items()}, name='levels kept').to_frame()

359 built features -> 372 after one-hot -> 114 candidates after the pre-filter


,features
dropped_because,
correlated > 0.95,227
non-zero for fewer than 3 players or constant,31


,levels kept
COUNTRY,"United States, Other"
REGION,"Europe, LATAM, N-America, Other"
ATTRIBUTION_TYPE,"new, new_game, Other"
SOURCE_FIRST,"adwords, facebook, organic, Other"
SOURCE_TYPE_FIRST,"organic, ua, Other"
MAIN_PLATFORM_ID,"android, ios, Other"


The pre-filter keeps a feature if at least 3 training players have a non-zero value (revenue features are non-zero for only 7 players, so a share-based rule would remove them), then keeps one of each pair with rank correlation above 0.95.

**Leakage checks** (all must pass before training):
1. *Truncation:* features built from data cut at each player's feature date equal features built from the full file.
2. *Future revenue:* no revenue-type feature column shows more than the player's revenue so far on its row.
3. *Consistency:* our revenue over days 0–2 / 0–6 equals `I3_REVENUE` / `I7_REVENUE` read inside their windows.

In [12]:
rev_cols = [c for c in spec.count + spec.state + spec.average if 'REVENUE' in c]
print('1. Truncation check passed:', checks.truncation_check(pday, installs, spec))
print('2. Future revenue check:')
display(checks.future_revenue_check(pday, installs, rev_cols))
print('3. Consistency with I3/I7_REVENUE:')
display(checks.install_window_consistency(pday, installs))

1. Truncation check passed: True
2. Future revenue check:


,column,rows_checked,rows_ahead
0,BUY_COINS_REVENUE,0,0
1,HOT_OFFER_REVENUE,7,0
2,OOC_REVENUE,9,0
3,REVENUE,14,0
4,DTC_REVENUE,14,0
5,DTC_NET_FEE_REVENUE,14,0


3. Consistency with I3/I7_REVENUE:


,column,players,match_share,mismatches
0,I3_REVENUE,700,1.000000,0
1,I7_REVENUE,700,0.998571,1


`I14_REVENUE` and `I30_REVENUE` are excluded outright: on early rows they already contain purchases made later (player 8520 shows $15.40 on install day; the purchase was on day 1–2). `I3`/`I7` are equal to our own sums, so they add nothing as features and serve only as this check.

## 7. Model selection on the training folds

Two candidates through the **same** pipeline (features, pre-filter, rounds, SHAP): **LightGBM** (Tweedie), the model you would run at scale, and a **regularised Tweedie GLM**, a simpler challenger. Each runs 4 rounds (all → 50 → 20 → 10 features by mean |SHAP|) × 8 settings on two forward folds (May → June, May–June → July), scored by Tweedie deviance on the remaining revenue (days 3–29). The test set is not used.

Device, app and performance columns (load times, latency, FPS, spin length, device type, OS version) were removed from the LTV candidates before this run: they describe the phone and app, not the player's intent to spend; with 6 payers the first run learned "slower loading → higher value" from them. They stay in the Part 4 friction list. *Transparency:* a diagnostic on the test set was looked at when discussing this; nothing else was chosen from it.

Only 4 of the 200 validation players spent anything in days 3–29, so read differences with care.

In [13]:
import json
from sciplay.ltv import learners as L, train as T
root = REPO_ROOT / 'models' / 'ltv'
best = pd.read_json(root / 'selection' / 'best_rounds.json')
cols = ['model', 'round', 'n_features', 'deviance', 'MAE_30d', 'pred_over_actual_30d', 'dollar_recall_30d', 'payer_recall_30d', 'picked']
display(best[cols].round(3))
print('Winner by the rule set in advance (lowest fold deviance; within 5% the simpler model):', T.pick_winner(best))

/sessions/rcw-01f6ypbpbm4vcdroqndkzs5x/venv-check/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,model,round,n_features,deviance,MAE_30d,pred_over_actual_30d,dollar_recall_30d,payer_recall_30d,picked
0,LightGBM,1,114,2.950,0.473,0.824,0.628,0.5,False
1,LightGBM,2,21,2.320,0.433,0.756,0.628,0.5,False
2,LightGBM,3,18,2.184,0.425,0.740,0.628,0.5,True
3,LightGBM,4,10,2.646,0.487,0.849,0.628,0.5,False
4,Tweedie GLM,1,114,4.099,0.938,1.741,0.628,0.5,False
5,Tweedie GLM,2,50,4.015,0.981,1.835,0.628,0.5,False
6,Tweedie GLM,3,20,3.087,0.947,1.747,0.628,0.5,False
7,Tweedie GLM,4,10,2.445,0.596,1.078,0.628,0.5,True


Winner by the rule set in advance (lowest fold deviance; within 5% the simpler model): lightgbm


**Decision: LightGBM** (deviance 2.18 vs 2.44 for the GLM, 12% apart, beyond the 5% tie margin), with 18 features (the 10-feature round scored clearly worse). With 4 validation payers the gap is within noise; the simpler GLM was close. The pipeline swaps models with one setting. *With more data, a smarter tuning search (e.g. Optuna) would replace the small fixed grid; here more trials would only fit noise.*

In [14]:
meta = json.loads((root / json.loads((root / 'active.json').read_text())['version'] / 'meta.json').read_text())
art_dir = root / meta['version']
print('Shipped:', meta['model'], '|', len(meta['features']), 'features |', meta['setting'], '|', meta['n_estimators'], 'trees')
print('Model version:', meta['version'])
pd.Series(meta['features'], name='features').to_frame()

Shipped: LightGBM | 18 features | {'num_leaves': 4, 'min_child_samples': 40, 'tweedie_variance_power': 1.2} | 135 trees
Model version: ltv-d3-lightgbm-6008a0b5-729fa670


,features
0,revenue_first_k
1,MAX_BET_AMOUNT__mean
2,MIN_BETS__mean
3,DAILY_JEWEL_TASKS_COMPLETED__per_day
4,MAX_BETS__mean
5,NUM_AUTO_SPIN__sum
6,DAILY_JEWEL_TASKS_COMPLETED__max
7,MAX_BANKROLL__last
8,TOTAL_WARNING_ERROR__sum
9,BET_DENOM__mean


## 8. Test results

Every approach fitted on train and graded **once** on the 218 test players (6 payers). The 30-day forecast = revenue in days 0–2 + predicted days 3–29. Ranges: the test players redrawn 1,000 times (model fixed), 5th–95th percentile. Dollar recall = share of actual revenue from players predicted at $1 or more.

In [15]:
res = pd.read_csv(art_dir / 'test_results.csv')
res['MAE range'] = res.apply(lambda r: f"${r.MAE_low:.2f}–{r.MAE_high:.2f}", axis=1)
res['ratio range'] = res.apply(lambda r: f"{r.ratio_low:.2f}–{r.ratio_high:.2f}", axis=1)
res[['approach', 'MAE', 'MAE range', 'MAE_payers', 'pred_over_actual', 'ratio range', 'dollar_recall', 'payer_recall', 'deviance']].round(3)

,approach,MAE,MAE range,MAE_payers,pred_over_actual,ratio range,dollar_recall,payer_recall,deviance
0,Baseline: revenue so far,1.149,$0.17–2.73,41.735,0.338,0.22–0.58,0.942,0.667,401.602
1,Baseline: training average,3.042,$1.59–5.12,61.651,0.798,0.36–5.47,1.000,1.000,9.137
2,Benchmark: simple rule,1.799,$0.87–3.33,39.556,0.788,0.50–3.11,0.942,0.667,3.743
3,LightGBM (shipped),1.174,$0.23–2.77,39.693,0.432,0.29–0.88,0.942,0.667,3.465
4,Tweedie GLM (challenger),1.443,$0.46–3.06,41.192,0.533,0.36–1.66,0.942,0.667,2.763


In [16]:
print('Value bands (predicted 30-day revenue):')
display(pd.read_csv(art_dir / 'value_bands.csv'))
print('By install month:')
display(pd.read_csv(art_dir / 'test_by_month.csv').round(2))

Value bands (predicted 30-day revenue):


,band,players,payers,predicted_total,actual_total,actual_per_player
0,$0–1,212,2,8.59,21.89,0.10
1,$1–20,3,1,23.50,10.96,3.65
2,$20+,3,3,131.22,345.36,115.12


By install month:


,month,players,payers,predicted,actual
0,Aug,100,2,76.22,95.44
1,Sep,100,3,34.72,57.43
2,Oct,18,1,52.38,225.34


**Headline:** the day-3 forecast **ranks** players well: the 3 players it values at $20+ brought in 91% of test revenue ($345 of $378), the 212 it values under $1 brought in 6%. But with only 6 paying test players it **cannot beat simply "revenue so far" on error per player** ($1.17 vs $1.15; all ranges overlap), and like every approach it **under-predicts total dollars** (0.43): one player spent $50.80 by day 3 and $225 in 30 days. It did not catch the 2 test players who first paid after day 3. **Use it to rank and prioritise players now; trust its dollar levels once more payers are available** (maturity table, section 2 of the docs).

Why not just use revenue so far? It gives $0 to every player who has not paid yet (97%), so it cannot rank them.

## 9. Explanations (SHAP) and reason codes

Global importance = mean |SHAP| of the shipped model on the training players (both directions count).

**Player reason codes:** the features with the largest |SHAP| for that player, until they cover 70% of the player's total |SHAP| (at most 3, none under 10%). Each reads *"feature: this player's value vs typical (median training player) (±$ effect)"*. The $ effect is how much lower (or higher) the forecast would be **without that one signal**, all else equal: the model works on a log scale, so contributions multiply and removing one divides the forecast by exp(SHAP). Effects overlap, so they do not add up to the forecast. Players with no effect above $0.05 get *"No strong value signals in the first 3 days"* with the average new player's value for comparison.

In [17]:
import joblib
from sciplay.ltv import score as S
art = joblib.load(art_dir / 'model.joblib')
dd = T.load_data()
tr_ids = dd.sets.index[dd.sets == 'train']
Xtr = art['encoder'].transform(dd.X_raw.loc[tr_ids].drop(columns='feature_date')).reindex(columns=art['features'])
sv = L.LEARNERS[art['learner']].shap_values(art['model'], Xtr)
imp = pd.Series(abs(sv).mean(axis=0), index=[S.feature_label(f) for f in art['features']]).sort_values()
fig = go.Figure(go.Bar(x=imp.values, y=imp.index, orientation='h', marker_color='#6B3FA0',
                       hovertemplate='%{y}: %{x:.3f}<extra></extra>'))
fig.update_layout(title='What drives the forecast: mean |SHAP| (log scale), training players', template='plotly_white',
                  height=560, margin=dict(l=380), xaxis_title=None)
fig.show()

In [18]:
scores = S.read('2026-11-04')
m = scores[scores.source == 'model_d3'].sort_values('predicted_ltv_365', ascending=False)
m[['player_id', 'age_days', 'ltv_to_date', 'predicted_ltv_30', 'predicted_ltv_365', 'reason_1', 'reason_2', 'reason_3']].head(5).round(2)

,player_id,age_days,ltv_to_date,predicted_ltv_30,predicted_ltv_365,reason_1,reason_2,reason_3
571,287726,24,0.00,12.89,32.16,"Regular spin, first 3 days: 6,365 vs typical 217 (+$9.25)","Num auto spin, first 3 days: 490 vs typical 0 (+$8.78)","Regular spin, per day played: 2,121.67 vs typical 149.25 (+$8.19)"
631,301504,3,12.05,12.22,30.49,"Daily jewel tasks completed, per day played: 6.03 vs typical 2.94 (+$0.15)","Max bankroll, on day 3: 657.1B vs typical 9.6B (+$0.13)","Max bet amount, average: 42.2B vs typical 154.3M (+$0.10)"
537,274034,20,0.00,8.45,21.08,"Num auto spin, first 3 days: 6,679 vs typical 0 (+$6.15)","Regular spin, first 3 days: 7,301 vs typical 217 (+$6.07)","Pearl inflow coins, first 3 days: 18.6B vs typical 101.5M (+$5.55)"
546,279949,13,0.00,7.13,17.79,"Num auto spin, first 3 days: 8,357 vs typical 0 (+$5.19)","Regular spin, first 3 days: 9,375 vs typical 217 (+$5.12)","Daily jewel tasks completed, per day played: 6.2 vs typical 2.94 (+$4.97)"
589,293858,18,0.00,3.65,9.11,"Num auto spin, first 3 days: 2,969 vs typical 0 (+$2.69)","Regular spin, first 3 days: 4,578 vs typical 217 (+$2.68)","Pearl inflow coins, first 3 days: 4.6B vs typical 101.5M (+$2.36)"


## 10. Uncertainty and confidence

**Range (bootstrap):** the shipped model is retrained 50 times, each time on the 300 training players drawn with replacement; every player gets 50 forecasts. The spread shows how much a forecast depends on which players happened to be in training. The shown range is the forecast × (25th or 75th percentile ÷ median of the 50), so it keeps the relative spread and always contains the forecast. LTV365 range = 30-day range × 2.49.

**Why the confidence label is rule-based for now:** with 6 payers most resampled models predict close to $0 for nearly everyone, so they agree; the spread could not separate good forecasts from bad ones (the first run put all 6 test payers in the 'High' group). With thousands of payers the spread becomes a real, checkable measure (low confidence should mean larger errors). Until then: **High** = paid in the first 3 days (the forecast rests on real spending); **Medium** = active non-payer; under 3 days old = no forecast.

In [19]:
pd.read_csv(art_dir / 'confidence_check.csv').round(2)

,confidence,players,payers,MAE,predicted,actual
0,High,4,4,54.23,144.98,356.32
1,Medium,214,2,0.18,18.34,21.89


## 11. Players too old for the model: the update rule

Players installed before 7 Oct are older than the model's window. They get a rule, **not the LTV model**: revenue to date + last 30 days' revenue × (r + r² + …) over the 30-day blocks left until day 365, with the same assumed r = 0.6; nothing more if not seen for 30+ days. Backtest of its next-30-day part at past dates:

In [20]:
A.update_rule_backtest(pday, installs)

,cutoff,approach,players,payers_next_30d,MAE_$,predicted_$,actual_$,pred_over_actual
0,2026-08-01,Update rule (last 30 days x r),206,5,1.92,598.0,757.0,0.79
1,2026-08-01,Last 30 days repeated,206,5,2.06,997.0,757.0,1.32
2,2026-08-01,Zero,206,5,3.67,0.0,757.0,0.00
3,2026-09-01,Update rule (last 30 days x r),305,3,1.19,408.0,638.0,0.64
4,2026-09-01,Last 30 days repeated,305,3,0.61,679.0,638.0,1.06
5,2026-09-01,Zero,305,3,2.09,0.0,638.0,0.00
6,2026-10-01,Update rule (last 30 days x r),404,4,0.56,405.0,299.0,1.35
7,2026-10-01,Last 30 days repeated,404,4,1.22,674.0,299.0,2.26
8,2026-10-01,Zero,404,4,0.74,0.0,299.0,0.00


The rule is in the right range (0.64–1.35 of actual) and closer than repeating the last 30 days at two of three dates. It is labelled "not from the LTV model" everywhere it appears.

## 12. The output table

`ltv_scores` in SQLite, one row per player per score date, keyed on (score date, player, model version) so reruns replace rather than duplicate. Run `python -m sciplay.ltv.score --date 2026-11-04`.

In [21]:
display(scores.groupby('source').agg(players=('player_id', 'size'), ltv_to_date=('ltv_to_date', 'sum'),
                                     predicted_ltv_365=('predicted_ltv_365', 'sum')).round(0))
scores[scores.source == 'model_d3'].head(3).T

,players,ltv_to_date,predicted_ltv_365
source,,,
cold_start,50,0.0,0.0
model_d3,132,12.0,132.0
update_rule,518,4176.0,4886.0


,500,501,502
score_date,2026-11-04,2026-11-04,2026-11-04
player_id,259371,259937,260895
model_version,ltv-d3-lightgbm-6008a0b5-729fa670,ltv-d3-lightgbm-6008a0b5-729fa670,ltv-d3-lightgbm-6008a0b5-729fa670
feature_date,2026-10-09,2026-10-30,2026-10-29
install_date,2026-10-07,2026-10-28,2026-10-27
age_days,28,7,8
source,model_d3,model_d3,model_d3
eligibility_status,eligible,eligible,eligible
ltv_to_date,0.0,0.0,0.0
predicted_ltv_30,0.001059,0.005961,0.279021
